<div style="border-left: 6px solid #00356B; padding-left: 15px; margin-bottom: 20px;">
  <h1 style="margin-bottom: 5px; color: #00356B"><strong>Assignment 3:</strong> Part 2 (Self-Play with One Model)</h1>
  <span style="font-size: 1.2em; color: #444; font-weight: bold">S&DS 5350 | Social Algorithms</span>
  <br><br>
  <strong>Primary:</strong> Brandon Tran (bat53)
  <br>
  <strong>Partner:</strong> Cailey Bobadilla (cjb239)
  <br>
  <strong>Group:</strong> 9
</div>

---

*Mood for this part:*

<iframe data-testid="embed-iframe" style="border-radius:12px" src="https://open.spotify.com/embed/track/1DwscornXpj8fmOmYVlqZt?utm_source=generator" width="40%" height="152" frameBorder="0" allowfullscreen="" allow="autoplay; clipboard-write; encrypted-media; fullscreen; picture-in-picture" loading="lazy"></iframe>

In [ ]:
%pip install pandas

Now let one model play against itself in a 2-player Scattergories game with many questions.

Use the provided question bank:
- `assets/assignment3/scattergories_questions.csv`

#### II.1 Game definition

Use a two-phase process.

Phase A: generate answers

1. For each row `(letter, category)` and each round index, have the player output one answer. Write a prompt that wraps around the `category` and `letter` from `scattergories_questions.csv`, and set the temperature informed by your experiences above. Run both instances of the model (player 1 and player 2) with the same prompt and same temperature (optional: explore varying temperature here as well).
2. Write the answers to a CSV file in the required format.
3. Keep this generation step independent from judging.

Phase A was performed using `part2_phaseA.py` and `assignment3_starter.py`.

Include qwen model, temperature, top_k, number of rounds

In [ ]:
import pandas as pd

# Read in CSV files for player 1 and player 2 at temperature=0.9
df_default_p1 = pd.read_csv("outputs/default_09_Player_1.csv")
df_default_p2 = pd.read_csv("outputs/default_09_Player_2.csv")

# Read in CSV files for player 1 and player 2 at temperature=2.0
df_opt_p1 = pd.read_csv("outputs/opt_20_Player_1.csv")
df_opt_p2 = pd.read_csv("outputs/opt_20_Player_2.csv")

In [ ]:
# Display the head of the answers for player 1 at temperature=0.9
df_default_p1.head()

,question_id,letter,category,round_idx,answer,model,player_id,temperature,top_k,prompt_id
0,Q001,A,Animals,0,Antelope,qwen2.5:7b,Player_1_default,0.9,40,baseline
1,Q001,A,Animals,1,Albatross,qwen2.5:7b,Player_1_default,0.9,40,baseline
2,Q001,A,Animals,2,Ape,qwen2.5:7b,Player_1_default,0.9,40,baseline
3,Q001,A,Animals,3,Albatross,qwen2.5:7b,Player_1_default,0.9,40,baseline
4,Q001,A,Animals,4,Antelope,qwen2.5:7b,Player_1_default,0.9,40,baseline


In [ ]:
# Display the head of the answers for player 2 at temperature=0.9
df_default_p2.head()

,question_id,letter,category,round_idx,answer,model,player_id,temperature,top_k,prompt_id
0,Q001,A,Animals,0,Albatross,qwen2.5:7b,Player_2_default,0.9,40,baseline
1,Q001,A,Animals,1,Albatross,qwen2.5:7b,Player_2_default,0.9,40,baseline
2,Q001,A,Animals,2,Alpaca,qwen2.5:7b,Player_2_default,0.9,40,baseline
3,Q001,A,Animals,3,Antelope,qwen2.5:7b,Player_2_default,0.9,40,baseline
4,Q001,A,Animals,4,Antelope,qwen2.5:7b,Player_2_default,0.9,40,baseline


In [ ]:
# Display the head of the answers for player 1 at temperature=2.0
df_opt_p1.head()

,question_id,letter,category,round_idx,answer,model,player_id,temperature,top_k,prompt_id
0,Q001,A,Animals,0,Albatross,qwen2.5:7b,Player_1_opt,2.0,40,baseline
1,Q001,A,Animals,1,Aardvark,qwen2.5:7b,Player_1_opt,2.0,40,baseline
2,Q001,A,Animals,2,Aardvark,qwen2.5:7b,Player_1_opt,2.0,40,baseline
3,Q001,A,Animals,3,Antelope,qwen2.5:7b,Player_1_opt,2.0,40,baseline
4,Q001,A,Animals,4,Albatross,qwen2.5:7b,Player_1_opt,2.0,40,baseline


In [ ]:
# Display the head of the answers for player 2 at temperature=2.0
df_opt_p2.head()

,question_id,letter,category,round_idx,answer,model,player_id,temperature,top_k,prompt_id
0,Q001,A,Animals,0,Antelope,qwen2.5:7b,Player_2_opt,2.0,40,baseline
1,Q001,A,Animals,1,Aardvark,qwen2.5:7b,Player_2_opt,2.0,40,baseline
2,Q001,A,Animals,2,Albatross,qwen2.5:7b,Player_2_opt,2.0,40,baseline
3,Q001,A,Animals,3,Alpaca,qwen2.5:7b,Player_2_opt,2.0,40,baseline
4,Q001,A,Animals,4,Antelope,qwen2.5:7b,Player_2_opt,2.0,40,baseline


Phase B: judge and score

1. Run `judge.py` on one or more answer files. Reminder that will require your OpenAI API key.
2. The judge script will:
    - call a GPT judge for validity (`yes`/`no`)
    - normalize answers
    - compute points across submitted player files
    - output score CSV
3. Audit quality: randomly sample at least 50 judged examples and manually verify them; report estimated judge error rate.

In [16]:
import pandas as pd

# Read in CSV files for the judged rows from both scattergories games
df_default = pd.read_csv("outputs/judged_default.csv")
df_opt = pd.read_csv("outputs/judged_opt.csv")

# Combine the dataframes into one bigger dataframe
df_all_judged = pd.concat([df_default, df_opt])

# Randomly sample 50 rows for manual verification
audit_sample = df_all_judged.sample(n=50, random_state=42)

# Select the necessary columns for manual verification
audit_sample = audit_sample[['letter', 'category', 'answer_norm', 'valid']]

# Add a blank column for manual verification
audit_sample['manual_verification'] = ""

# Save the sample of 50 to a CSV
audit_sample.to_csv("outputs/audit_quality.csv", index=False)

In [17]:
import pandas as pd

# Read in CSV file for the manual verification of the sampled judged rows
df_audit_sample = pd.read_csv("outputs/audit_quality.csv")

# Display the head of the manual verification of the sampled judged rows
df_audit_sample.head()

,letter,category,answer_norm,valid,manual_verification
0,E,Companies,etsy,1,1
1,F,Things in a backpack,flashlight,1,1
2,Z,Things in a zoo,zookeeper,1,1
3,N,Birds,nightjar,1,1
4,K,Superheroes,kirby,0,0


In [24]:
# Create a series of True and False values based on if columns match in value
matches = df_audit_sample['valid'] == df_audit_sample['manual_verification']

# Calculate the proportion of correct matches
accuracy = matches.mean()

# Print the estimated accuracy and error rates
print(f"Estimated Judge Accuracy Rate: {round(accuracy * 100, 1)}%")
print(f"Estimated Judge Error Rate: {round((1 - accuracy) * 100, 1)}%")

Estimated Judge Accuracy Rate: 92.0%
Estimated Judge Error Rate: 8.0%


Wrote outputs/judged_default.csv
Wrote outputs/scores_default.csv
Judge API calls: 209
Judge cache hits: 430

Wrote outputs/judged_opt.csv
Wrote outputs/scores_opt.csv
Judge API calls: 147
Judge cache hits: 492

#### II.2 Self-play experiments

1. Once you have generation and judging figured out, run repeated rounds for each question (enough rounds for stable estimates (of the expected score) and store generated answers).
2. Run `judge.py` on your generated files to compute game outcomes from self-play.
3. Measure per-question and overall outcomes:
    - Validity rate
    - Average score per player
4. Revisit prompt/temperature choices and see whether you can improve the self-play score.

In [ ]:
import pandas as pd

